# Phase 3 — Component and Depth Compression Sensitivity

Systematic study of selective SVD compression on model quality and on the
structure of the embedding space.  Two orthogonal axes are analysed:

1. **Per component**: Query, Key, Value, Attention Output, FFN Intermediate,
   FFN Output, full Attention, full FFN.
2. **Per depth group**: early (0-3), middle (4-7), late (8-11).

Each configuration is evaluated at three SVD ranks (256, 128, 64).

- **Metrics**: F1 macro (GoEmotions test) + silhouette score on joint t-SNE clusters.
- **Part A**: setup and baseline.
- **Part B**: component sensitivity study.
- **Part C**: depth sensitivity study.
- **Part D**: dashboards and summary.
- **Part E**: CSV export to `results/notebook3/`.

## 0. Setup

In [ ]:
import sys
import os

# Project root (works both on Colab and locally).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    count_parameters,
    get_compression_ratio,
    get_target_layer_names,
    filter_layer_names,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['font.size'] = 11

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ---- Experiment configuration -------------------------------------------
MODEL_SUBDIR = 'bert-goemotions-23emo-final'
EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']

# SVD ranks tested in both studies.
RANKS = [256, 128, 64]

# Components tested in the per-component study.
COMPONENTS = [
    ('query',            'Query'),
    ('key',              'Key'),
    ('value',            'Value'),
    ('attention_output', 'Attn_Output'),
    ('intermediate',     'FFN_Inter'),
    ('ffn_output',       'FFN_Output'),
    ('attention',        'Attention_All'),
    ('ffn',              'FFN_All'),
]

# Depth groups tested in the per-depth study.
DEPTH_GROUPS = [
    (list(range(0, 4)),  'Early (0-3)'),
    (list(range(4, 8)),  'Middle (4-7)'),
    (list(range(8, 12)), 'Late (8-11)'),
]

# t-SNE: subsample size (larger is slower but more stable).
N_SAMPLES = 1500
TSNE_SEED = 42

# ---- Paths --------------------------------------------------------------
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
MODEL_PATH = os.path.join(RESULTS_DIR, MODEL_SUBDIR)
FIGURES_DIR = RESULTS_DIR
EXPORT_DIR = os.path.join(RESULTS_DIR, 'notebook3')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f'Model path:  {MODEL_PATH}')
print(f'Figures dir: {FIGURES_DIR}')
print(f'Export dir:  {EXPORT_DIR}')

In [ ]:
# ---- Load model and dataset --------------------------------------------
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, problem_type='multi_label_classification',
)
num_labels = baseline_model.config.num_labels
baseline_model.eval().to(device)
print(f'num_labels = {num_labels}')
print(f'Total parameters: {count_parameters(baseline_model)["total"]:,}')

_, _, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS,
)
print(f'Test set: {len(test_ds)} examples')

# Deterministic subsample of the test set for t-SNE.
rng = np.random.RandomState(TSNE_SEED)
sample_indices = rng.choice(len(test_ds), size=min(N_SAMPLES, len(test_ds)), replace=False)
sample_indices.sort()
sub_test = test_ds.select(sample_indices)
print(f'Subsample: {len(sub_test)} examples')

all_target_names = get_target_layer_names(baseline_model)
print(f'Compressible layers: {len(all_target_names)}')

## 1. Helpers

In [ ]:
eval_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, 'tmp_eval'),
    per_device_eval_batch_size=64,
    fp16=(device == 'cuda'),
    report_to='none',
)


def evaluate_model(model):
    '''Evaluate a model on the test set and return the metrics dict.'''
    trainer = Trainer(
        model=model, args=eval_args,
        compute_metrics=compute_metrics, data_collator=data_collator,
    )
    return trainer.evaluate(test_ds)


@torch.no_grad()
def extract_cls_embeddings(model, dataset, batch_size=64):
    '''Extract [CLS] embeddings from the last hidden state.'''
    model.eval()
    model_device = next(model.parameters()).device

    all_embeddings, all_labels = [], []
    for i in range(0, len(dataset), batch_size):
        batch_indices = range(i, min(i + batch_size, len(dataset)))
        batch = data_collator([dataset[j] for j in batch_indices])
        input_ids = batch['input_ids'].to(model_device)
        attention_mask = batch['attention_mask'].to(model_device)
        labels = batch['labels'].numpy()
        outputs = model(
            input_ids=input_ids, attention_mask=attention_mask,
            output_hidden_states=True,
        )
        cls_emb = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        all_embeddings.append(cls_emb)
        all_labels.append(labels)

    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)


def run_joint_tsne(embeddings_dict, perplexity=30, n_iter=1000, seed=TSNE_SEED):
    '''Run t-SNE on concatenated embeddings of several models.

    Returns tsne_per_model (name -> coords) and shared bounds (for plotting).
    '''
    model_names = list(embeddings_dict.keys())
    all_embs = np.concatenate([embeddings_dict[n] for n in model_names], axis=0)
    n_per_model = len(embeddings_dict[model_names[0]])
    print(f'  Total points: {all_embs.shape[0]} ({len(model_names)} models x {n_per_model})')

    tsne = TSNE(
        n_components=2, perplexity=perplexity, n_iter=n_iter,
        random_state=seed, init='pca', learning_rate='auto',
    )
    all_coords = tsne.fit_transform(all_embs)

    tsne_per_model = {
        name: all_coords[i * n_per_model:(i + 1) * n_per_model]
        for i, name in enumerate(model_names)
    }
    bounds = (
        all_coords[:, 0].min() - 2, all_coords[:, 0].max() + 2,
        all_coords[:, 1].min() - 2, all_coords[:, 1].max() + 2,
    )
    return tsne_per_model, bounds


def compute_silhouette_scores(tsne_per_model, dominant_emotion, top_indices):
    '''Silhouette score per model using only the top dominant emotions as classes.'''
    top_mask = np.isin(dominant_emotion, top_indices)
    top_labels = dominant_emotion[top_mask]
    scores = {}
    for name, coords in tsne_per_model.items():
        top_coords = coords[top_mask]
        scores[name] = silhouette_score(
            top_coords, top_labels,
            sample_size=min(1000, len(top_coords)),
            random_state=TSNE_SEED,
        )
    return scores


def create_compressed_model(baseline, rank, component=None, layers=None):
    '''Build an SVD-compressed copy of `baseline` restricted to a component/depth subset.

    Workaround for the known filter_layer_names bug: component="ffn_output"
    also matches attention.output.dense, so we filter those out here.
    '''
    target_names = filter_layer_names(
        get_target_layer_names(baseline), component=component, layers=layers,
    )
    if component == 'ffn_output':
        target_names = [n for n in target_names if 'attention' not in n]

    compressed = apply_svd_compression(baseline, rank=rank, layer_names=target_names)
    return compressed, target_names

In [ ]:
def plot_scatter_panel(tsne_per_model, bounds, dominant_emotion, top_indices,
                       f1_scores, ncols=3, title='', save_path=None):
    '''Grid of scatter plots coloured by dominant emotion.'''
    model_names = list(tsne_per_model.keys())
    n_models = len(model_names)
    nrows = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 6))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes[np.newaxis, :]
    elif ncols == 1:
        axes = axes[:, np.newaxis]

    x_min, x_max, y_min, y_max = bounds
    cmap = plt.cm.tab10
    is_top = np.isin(dominant_emotion, top_indices)

    for idx, name in enumerate(model_names):
        row, col = divmod(idx, ncols)
        ax = axes[row, col]
        coords = tsne_per_model[name]

        ax.scatter(
            coords[~is_top, 0], coords[~is_top, 1],
            c=[(0.85, 0.85, 0.85, 0.2)], s=3, rasterized=True,
        )
        for rank_i, emo_idx in enumerate(top_indices):
            mask = dominant_emotion == emo_idx
            ax.scatter(
                coords[mask, 0], coords[mask, 1],
                c=[cmap(rank_i)], s=8, alpha=0.6,
                label=emotion_names[emo_idx] if idx == 0 else None,
                rasterized=True,
            )

        f1 = f1_scores.get(name, 0)
        ax.set_title(f'{name}\nF1 macro = {f1:.4f}', fontsize=11, fontweight='bold')
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xticks([])
        ax.set_yticks([])

    for idx in range(n_models, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row, col].set_visible(False)

    axes[0, 0].legend(
        bbox_to_anchor=(-0.15, 0.5), loc='center right',
        fontsize=8, frameon=True, framealpha=0.9, markerscale=2.5,
    )
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


def plot_kde_panel(tsne_per_model, bounds, dominant_emotion, top_indices,
                   f1_scores, ncols=3, n_kde_emotions=6, title='', save_path=None):
    '''Grid of KDE contour plots per emotion.'''
    model_names = list(tsne_per_model.keys())
    n_models = len(model_names)
    nrows = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 6))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes[np.newaxis, :]
    elif ncols == 1:
        axes = axes[:, np.newaxis]

    x_min, x_max, y_min, y_max = bounds
    cmap = plt.cm.tab10
    top_for_kde = top_indices[:n_kde_emotions]

    for idx, name in enumerate(model_names):
        row, col = divmod(idx, ncols)
        ax = axes[row, col]
        coords = tsne_per_model[name]

        ax.scatter(coords[:, 0], coords[:, 1],
                   c=[(0.9, 0.9, 0.9, 0.15)], s=1, rasterized=True)

        for rank_i, emo_idx in enumerate(top_for_kde):
            mask = dominant_emotion == emo_idx
            if mask.sum() < 10:
                continue
            x = coords[mask, 0]
            y = coords[mask, 1]
            try:
                kde = gaussian_kde(np.vstack([x, y]), bw_method=0.3)
                xi = np.linspace(x_min, x_max, 100)
                yi = np.linspace(y_min, y_max, 100)
                xi, yi = np.meshgrid(xi, yi)
                zi = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)
                color = cmap(rank_i)
                ax.contour(xi, yi, zi, levels=3, colors=[color],
                          linewidths=1.5, alpha=0.7)
                ax.contourf(xi, yi, zi, levels=3, colors=[
                    (*color[:3], 0.05), (*color[:3], 0.1),
                    (*color[:3], 0.15), (*color[:3], 0.2),
                ])
            except np.linalg.LinAlgError:
                pass

        f1 = f1_scores.get(name, 0)
        ax.set_title(f'{name}\nF1 macro = {f1:.4f}', fontsize=11, fontweight='bold')
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xticks([])
        ax.set_yticks([])

    for idx in range(n_models, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row, col].set_visible(False)

    legend_handles = [
        Patch(facecolor=cmap(i), alpha=0.5, label=emotion_names[idx])
        for i, idx in enumerate(top_for_kde)
    ]
    axes[0, 0].legend(
        handles=legend_handles, bbox_to_anchor=(-0.15, 0.5), loc='center right',
        fontsize=8, frameon=True, framealpha=0.9,
    )
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()

---
## Part A — Baseline evaluation

In [ ]:
baseline_metrics = evaluate_model(baseline_model)
baseline_f1 = baseline_metrics['eval_f1_macro']
print(f'Baseline F1 macro: {baseline_f1:.4f}')

baseline_per_emotion = np.array(
    [baseline_metrics[f'eval_f1_label_{i}'] for i in range(num_labels)]
)

baseline_embs, baseline_labels = extract_cls_embeddings(baseline_model, sub_test)
print(f'Baseline embeddings: {baseline_embs.shape}')

dominant_emotion = np.argmax(baseline_labels, axis=1)
unique, counts = np.unique(dominant_emotion, return_counts=True)
top_indices = unique[np.argsort(-counts)][:10]
top_counts = counts[np.argsort(-counts)][:10]

print('\nTop 10 dominant emotions:')
for idx, count in zip(top_indices, top_counts):
    pct = count / len(dominant_emotion) * 100
    print(f'  {emotion_names[idx]:>15s}: {count:>4d} ({pct:5.1f}%)')

---
## Part B — Component sensitivity study

8 components x 3 ranks = 24 configurations.

| Component         | Description                                    |
|-------------------|------------------------------------------------|
| Query             | Query projections in self-attention            |
| Key               | Key projections in self-attention              |
| Value             | Value projections in self-attention            |
| Attn_Output       | Output projection of the attention block       |
| FFN_Inter         | FFN intermediate layer (768 -> 3072)           |
| FFN_Output        | FFN output layer (3072 -> 768)                 |
| Attention_All     | Q + K + V + Attn_Output (4 matrices x 12)      |
| FFN_All           | FFN_Inter + FFN_Output (2 matrices x 12)       |

### B1. Configuration

In [ ]:
print(f'Components: {len(COMPONENTS)}')
for comp_key, comp_label in COMPONENTS:
    names = filter_layer_names(all_target_names, component=comp_key)
    if comp_key == 'ffn_output':
        names = [n for n in names if 'attention' not in n]
    print(f'  {comp_label:<15s}: {len(names)} layers')
print(f'Ranks: {RANKS}')
print(f'Total models: {len(COMPONENTS) * len(RANKS)}')

### B2. Evaluate every (component, rank) configuration

In [ ]:
comp_results = []
comp_embeddings = {}
comp_per_emotion = {}

for comp_key, comp_label in COMPONENTS:
    for rank in RANKS:
        print('=' * 50)
        print(f'Component: {comp_label}, Rank: {rank}')

        model, target_names = create_compressed_model(
            baseline_model, rank, component=comp_key,
        )
        model.to(device)

        metrics = evaluate_model(model)
        f1 = metrics['eval_f1_macro']
        ratio = get_compression_ratio(baseline_model, model)
        embs, _ = extract_cls_embeddings(model, sub_test)

        comp_results.append({
            'component': comp_label,
            'component_key': comp_key,
            'rank': rank,
            'f1_macro': f1,
            'compression_ratio': ratio,
            'n_layers': len(target_names),
        })
        comp_embeddings[(comp_key, rank)] = embs
        comp_per_emotion[(comp_label, rank)] = np.array(
            [metrics[f'eval_f1_label_{i}'] for i in range(num_labels)]
        )

        print(f'  F1 macro: {f1:.4f}')
        print(f'  Compression ratio: {ratio:.2f}x')
        print(f'  Layers compressed: {len(target_names)}')

        del model
        if device == 'cuda':
            torch.cuda.empty_cache()

print('=' * 50)
print(f'Models evaluated: {len(comp_results)}')

### B3. Joint t-SNE and visualisations

For every rank we run one joint t-SNE over baseline + 8 component variants
(9 models x 1500 points = 13,500 points per run).

In [ ]:
comp_tsne = {}
comp_silhouette = {}

for rank in RANKS:
    print(f'\nt-SNE for rank {rank}...')
    emb_dict = {'Baseline': baseline_embs}
    for comp_key, comp_label in COMPONENTS:
        emb_dict[comp_label] = comp_embeddings[(comp_key, rank)]

    tsne_per_model, bounds = run_joint_tsne(emb_dict)
    comp_tsne[rank] = (tsne_per_model, bounds)

    sil_scores = compute_silhouette_scores(tsne_per_model, dominant_emotion, top_indices)
    for _, comp_label in COMPONENTS:
        comp_silhouette[(comp_label, rank)] = sil_scores[comp_label]
    comp_silhouette[('Baseline', rank)] = sil_scores['Baseline']

    print(f'  Silhouette Baseline: {sil_scores["Baseline"]:.4f}')
    for _, comp_label in COMPONENTS:
        print(f'  Silhouette {comp_label:<15s}: {sil_scores[comp_label]:.4f}')

In [ ]:
for rank in RANKS:
    tsne_per_model, bounds = comp_tsne[rank]
    f1_dict = {'Baseline': baseline_f1}
    for r in comp_results:
        if r['rank'] == rank:
            f1_dict[r['component']] = r['f1_macro']

    plot_scatter_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=3,
        title=f'Component compression -- Rank {rank}\n(scatter of dominant emotions, joint t-SNE)',
        save_path=os.path.join(FIGURES_DIR, f'comp_scatter_rank{rank}.png'),
    )

In [ ]:
for rank in RANKS:
    tsne_per_model, bounds = comp_tsne[rank]
    f1_dict = {'Baseline': baseline_f1}
    for r in comp_results:
        if r['rank'] == rank:
            f1_dict[r['component']] = r['f1_macro']

    plot_kde_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=3,
        title=f'Component compression -- Rank {rank}\n(KDE contours of dominant emotions, joint t-SNE)',
        save_path=os.path.join(FIGURES_DIR, f'comp_kde_rank{rank}.png'),
    )

In [ ]:
print('Silhouette scores -- Component study\n')
for rank in RANKS:
    print('=' * 45)
    print(f'Rank {rank}')
    print(f'{"Model":<18s} | {"Silhouette":>10s}')
    print('-' * 35)
    print(f'{"Baseline":<18s} | {comp_silhouette[("Baseline", rank)]:>10.4f}')
    for _, comp_label in COMPONENTS:
        print(f'{comp_label:<18s} | {comp_silhouette[(comp_label, rank)]:>10.4f}')
    print()

---
## Part C — Depth sensitivity study

3 depth groups x 3 ranks = 9 configurations.  All linear matrices
(Q, K, V, Attn_Output, FFN_Inter, FFN_Output) inside the selected layers
are compressed.

| Group        | Layers       | Description                                 |
|--------------|--------------|---------------------------------------------|
| Early (0-3)  | 0, 1, 2, 3   | Early layers — lexical / syntactic features |
| Middle (4-7) | 4, 5, 6, 7   | Middle layers — semantic features           |
| Late (8-11)  | 8, 9, 10, 11 | Late layers — task-specific features        |

### C1. Configuration

In [ ]:
print(f'Depth groups: {len(DEPTH_GROUPS)}')
for layer_indices, label in DEPTH_GROUPS:
    names = filter_layer_names(all_target_names, layers=layer_indices)
    print(f'  {label:<15s}: layers {layer_indices} ({len(names)} matrices)')
print(f'Ranks: {RANKS}')
print(f'Total models: {len(DEPTH_GROUPS) * len(RANKS)}')

### C2. Evaluate every (depth_group, rank) configuration

In [ ]:
depth_results = []
depth_embeddings = {}
depth_per_emotion = {}

for layer_indices, depth_label in DEPTH_GROUPS:
    for rank in RANKS:
        print('=' * 50)
        print(f'Depth: {depth_label}, Rank: {rank}')

        model, target_names = create_compressed_model(
            baseline_model, rank, layers=layer_indices,
        )
        model.to(device)

        metrics = evaluate_model(model)
        f1 = metrics['eval_f1_macro']
        ratio = get_compression_ratio(baseline_model, model)
        embs, _ = extract_cls_embeddings(model, sub_test)

        depth_results.append({
            'depth_group': depth_label,
            'layers': str(layer_indices),
            'rank': rank,
            'f1_macro': f1,
            'compression_ratio': ratio,
            'n_layers': len(target_names),
        })
        depth_embeddings[(depth_label, rank)] = embs
        depth_per_emotion[(depth_label, rank)] = np.array(
            [metrics[f'eval_f1_label_{i}'] for i in range(num_labels)]
        )

        print(f'  F1 macro: {f1:.4f}')
        print(f'  Compression ratio: {ratio:.2f}x')
        print(f'  Layers compressed: {len(target_names)}')

        del model
        if device == 'cuda':
            torch.cuda.empty_cache()

print('=' * 50)
print(f'Models evaluated: {len(depth_results)}')

### C3. Joint t-SNE and visualisations

For every rank we run one joint t-SNE over baseline + 3 depth groups
(4 models x 1500 points = 6,000 points per run).

In [ ]:
depth_tsne = {}
depth_silhouette = {}

for rank in RANKS:
    print(f'\nt-SNE for rank {rank}...')
    emb_dict = {'Baseline': baseline_embs}
    for _, depth_label in DEPTH_GROUPS:
        emb_dict[depth_label] = depth_embeddings[(depth_label, rank)]

    tsne_per_model, bounds = run_joint_tsne(emb_dict)
    depth_tsne[rank] = (tsne_per_model, bounds)

    sil_scores = compute_silhouette_scores(tsne_per_model, dominant_emotion, top_indices)
    for _, depth_label in DEPTH_GROUPS:
        depth_silhouette[(depth_label, rank)] = sil_scores[depth_label]
    depth_silhouette[('Baseline', rank)] = sil_scores['Baseline']

    print(f'  Silhouette Baseline: {sil_scores["Baseline"]:.4f}')
    for _, depth_label in DEPTH_GROUPS:
        print(f'  Silhouette {depth_label:<15s}: {sil_scores[depth_label]:.4f}')

In [ ]:
for rank in RANKS:
    tsne_per_model, bounds = depth_tsne[rank]
    f1_dict = {'Baseline': baseline_f1}
    for r in depth_results:
        if r['rank'] == rank:
            f1_dict[r['depth_group']] = r['f1_macro']

    plot_scatter_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=4,
        title=f'Depth compression -- Rank {rank}\n(scatter of dominant emotions, joint t-SNE)',
        save_path=os.path.join(FIGURES_DIR, f'depth_scatter_rank{rank}.png'),
    )

In [ ]:
for rank in RANKS:
    tsne_per_model, bounds = depth_tsne[rank]
    f1_dict = {'Baseline': baseline_f1}
    for r in depth_results:
        if r['rank'] == rank:
            f1_dict[r['depth_group']] = r['f1_macro']

    plot_kde_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=4,
        title=f'Depth compression -- Rank {rank}\n(KDE contours of dominant emotions, joint t-SNE)',
        save_path=os.path.join(FIGURES_DIR, f'depth_kde_rank{rank}.png'),
    )

In [ ]:
print('Silhouette scores -- Depth study\n')
for rank in RANKS:
    print('=' * 45)
    print(f'Rank {rank}')
    print(f'{"Model":<18s} | {"Silhouette":>10s}')
    print('-' * 35)
    print(f'{"Baseline":<18s} | {depth_silhouette[("Baseline", rank)]:>10.4f}')
    for _, depth_label in DEPTH_GROUPS:
        print(f'{depth_label:<18s} | {depth_silhouette[(depth_label, rank)]:>10.4f}')
    print()

---
## Part D — Summary dashboards

Comparative heatmaps of F1 macro and silhouette score for both studies.

In [ ]:
# ---- Build heatmap matrices -----------------------------------------------
comp_labels = [label for _, label in COMPONENTS]
depth_labels = [label for _, label in DEPTH_GROUPS]

f1_comp_matrix = np.zeros((len(COMPONENTS), len(RANKS)))
sil_comp_matrix = np.zeros((len(COMPONENTS), len(RANKS)))
for i, (_, comp_label) in enumerate(COMPONENTS):
    for j, rank in enumerate(RANKS):
        r = next(r for r in comp_results if r['component'] == comp_label and r['rank'] == rank)
        f1_comp_matrix[i, j] = r['f1_macro']
        sil_comp_matrix[i, j] = comp_silhouette[(comp_label, rank)]

f1_depth_matrix = np.zeros((len(DEPTH_GROUPS), len(RANKS)))
sil_depth_matrix = np.zeros((len(DEPTH_GROUPS), len(RANKS)))
for i, (_, depth_label) in enumerate(DEPTH_GROUPS):
    for j, rank in enumerate(RANKS):
        r = next(r for r in depth_results if r['depth_group'] == depth_label and r['rank'] == rank)
        f1_depth_matrix[i, j] = r['f1_macro']
        sil_depth_matrix[i, j] = depth_silhouette[(depth_label, rank)]

baseline_sil_per_rank = {r: comp_silhouette[('Baseline', r)] for r in RANKS}
baseline_sil_avg_comp = float(np.mean(list(baseline_sil_per_rank.values())))
baseline_sil_avg_depth = float(np.mean([depth_silhouette[('Baseline', r)] for r in RANKS]))
print('Heatmap matrices ready.')

In [ ]:
# ---- F1 heatmap (component) ---------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    f1_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=comp_labels,
    vmin=f1_comp_matrix.min() - 0.01, vmax=baseline_f1 + 0.005, ax=ax,
)
ax.set_xlabel('SVD rank', fontsize=12)
ax.set_ylabel('Component', fontsize=12)
ax.set_title(f'F1 macro per component and rank\n(Baseline: {baseline_f1:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'heatmap_f1_component.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ---- Silhouette heatmap (component) -------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    sil_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=comp_labels, ax=ax,
)
ax.set_xlabel('SVD rank', fontsize=12)
ax.set_ylabel('Component', fontsize=12)
ax.set_title(
    f'Silhouette score per component and rank\n(Baseline avg: {baseline_sil_avg_comp:.4f})',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'heatmap_silhouette_component.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ---- F1 heatmap (depth) -------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    f1_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=depth_labels,
    vmin=f1_depth_matrix.min() - 0.01, vmax=baseline_f1 + 0.005, ax=ax,
)
ax.set_xlabel('SVD rank', fontsize=12)
ax.set_ylabel('Depth group', fontsize=12)
ax.set_title(f'F1 macro per depth group and rank\n(Baseline: {baseline_f1:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'heatmap_f1_depth.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ---- Silhouette heatmap (depth) -----------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    sil_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=depth_labels, ax=ax,
)
ax.set_xlabel('SVD rank', fontsize=12)
ax.set_ylabel('Depth group', fontsize=12)
ax.set_title(
    f'Silhouette score per depth group and rank\n(Baseline avg: {baseline_sil_avg_depth:.4f})',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'heatmap_silhouette_depth.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ---- 2x2 combined dashboard ---------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

sns.heatmap(
    f1_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=comp_labels,
    vmin=f1_comp_matrix.min() - 0.01, vmax=baseline_f1 + 0.005,
    ax=axes[0, 0],
)
axes[0, 0].set_title(f'F1 macro -- Component\n(Baseline: {baseline_f1:.4f})', fontweight='bold')
axes[0, 0].set_xlabel('SVD rank'); axes[0, 0].set_ylabel('Component')

sns.heatmap(
    sil_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=comp_labels,
    ax=axes[0, 1],
)
axes[0, 1].set_title(f'Silhouette -- Component\n(Baseline: {baseline_sil_avg_comp:.4f})', fontweight='bold')
axes[0, 1].set_xlabel('SVD rank'); axes[0, 1].set_ylabel('')

sns.heatmap(
    f1_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=depth_labels,
    vmin=f1_depth_matrix.min() - 0.01, vmax=baseline_f1 + 0.005,
    ax=axes[1, 0],
)
axes[1, 0].set_title(f'F1 macro -- Depth\n(Baseline: {baseline_f1:.4f})', fontweight='bold')
axes[1, 0].set_xlabel('SVD rank'); axes[1, 0].set_ylabel('Depth group')

sns.heatmap(
    sil_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS], yticklabels=depth_labels,
    ax=axes[1, 1],
)
axes[1, 1].set_title(f'Silhouette -- Depth\n(Baseline: {baseline_sil_avg_depth:.4f})', fontweight='bold')
axes[1, 1].set_xlabel('SVD rank'); axes[1, 1].set_ylabel('')

fig.suptitle('Dashboard: impact of selective SVD compression',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dashboard_compression_analysis.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ---- Retention bar charts (F1 as % of baseline) -------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
colors = ['#2ecc71', '#f39c12', '#e74c3c']

comp_retention = (f1_comp_matrix / baseline_f1) * 100
x = np.arange(len(comp_labels))
width = 0.25
for j, rank in enumerate(RANKS):
    axes[0].bar(x + j * width, comp_retention[:, j], width,
                label=f'Rank {rank}', color=colors[j], alpha=0.85)
axes[0].axhline(y=100, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Baseline')
axes[0].set_xlabel('Component', fontsize=12)
axes[0].set_ylabel('F1 retention (%)', fontsize=12)
axes[0].set_title('F1 retention by component', fontsize=13, fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(comp_labels, rotation=45, ha='right')
axes[0].legend()
axes[0].set_ylim(0, 105)

depth_retention = (f1_depth_matrix / baseline_f1) * 100
x_d = np.arange(len(depth_labels))
for j, rank in enumerate(RANKS):
    axes[1].bar(x_d + j * width, depth_retention[:, j], width,
                label=f'Rank {rank}', color=colors[j], alpha=0.85)
axes[1].axhline(y=100, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Baseline')
axes[1].set_xlabel('Depth group', fontsize=12)
axes[1].set_ylabel('F1 retention (%)', fontsize=12)
axes[1].set_title('F1 retention by depth', fontsize=13, fontweight='bold')
axes[1].set_xticks(x_d + width)
axes[1].set_xticklabels(depth_labels, rotation=45, ha='right')
axes[1].legend()
axes[1].set_ylim(0, 105)

fig.suptitle('Performance retention under selective SVD compression',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'barchart_f1_retention.png'), bbox_inches='tight')
plt.show()

---
## Part E — Export results to `results/notebook3/`

In [ ]:
# ---- 1. Component sensitivity (master long table) -----------------------
df_comp = pd.DataFrame(comp_results)
df_comp['f1_retention'] = df_comp['f1_macro'] / baseline_f1
df_comp['f1_retention_pct'] = df_comp['f1_retention'] * 100
df_comp['silhouette'] = df_comp.apply(
    lambda row: comp_silhouette.get((row['component'], row['rank']), np.nan), axis=1,
)
df_comp.to_csv(os.path.join(EXPORT_DIR, 'component_sensitivity.csv'), index=False)
print(f'Saved: component_sensitivity.csv  {df_comp.shape}')

# ---- 2. Depth sensitivity (master long table) ---------------------------
df_depth = pd.DataFrame(depth_results)
df_depth['f1_retention'] = df_depth['f1_macro'] / baseline_f1
df_depth['f1_retention_pct'] = df_depth['f1_retention'] * 100
df_depth['silhouette'] = df_depth.apply(
    lambda row: depth_silhouette.get((row['depth_group'], row['rank']), np.nan), axis=1,
)
df_depth.to_csv(os.path.join(EXPORT_DIR, 'depth_sensitivity.csv'), index=False)
print(f'Saved: depth_sensitivity.csv  {df_depth.shape}')

# ---- 3. Heatmap matrices (wide format) ----------------------------------
pd.DataFrame(f1_comp_matrix, index=comp_labels,
             columns=[f'r{r}' for r in RANKS]).to_csv(
    os.path.join(EXPORT_DIR, 'component_f1_matrix.csv'), index_label='component')
pd.DataFrame(sil_comp_matrix, index=comp_labels,
             columns=[f'r{r}' for r in RANKS]).to_csv(
    os.path.join(EXPORT_DIR, 'component_silhouette_matrix.csv'), index_label='component')
pd.DataFrame(f1_depth_matrix, index=depth_labels,
             columns=[f'r{r}' for r in RANKS]).to_csv(
    os.path.join(EXPORT_DIR, 'depth_f1_matrix.csv'), index_label='depth_group')
pd.DataFrame(sil_depth_matrix, index=depth_labels,
             columns=[f'r{r}' for r in RANKS]).to_csv(
    os.path.join(EXPORT_DIR, 'depth_silhouette_matrix.csv'), index_label='depth_group')
print('Saved: component_f1_matrix.csv, component_silhouette_matrix.csv,')
print('       depth_f1_matrix.csv, depth_silhouette_matrix.csv')

# ---- 4. Baseline info ---------------------------------------------------
baseline_rows = [{
    'rank': r,
    'f1_macro': baseline_f1,
    'silhouette_comp_run': comp_silhouette[('Baseline', r)],
    'silhouette_depth_run': depth_silhouette[('Baseline', r)],
} for r in RANKS]
pd.DataFrame(baseline_rows).to_csv(
    os.path.join(EXPORT_DIR, 'baseline_info.csv'), index=False,
)
print('Saved: baseline_info.csv')

# ---- 5. Top emotions ----------------------------------------------------
top_rows = [{
    'rank_in_top': i,
    'emotion_index': int(idx),
    'emotion': emotion_names[idx],
    'count': int(count),
    'fraction': float(count / len(dominant_emotion)),
} for i, (idx, count) in enumerate(zip(top_indices, top_counts))]
pd.DataFrame(top_rows).to_csv(os.path.join(EXPORT_DIR, 'top_emotions.csv'), index=False)
print('Saved: top_emotions.csv')

# ---- 6. Sampled test points (for reproducibility of t-SNE) --------------
sample_rows = [{
    'point_idx': i,
    'test_index': int(sample_indices[i]),
    'dominant_emotion_idx': int(dominant_emotion[i]),
    'dominant_emotion': emotion_names[dominant_emotion[i]],
    'is_top_emotion': bool(dominant_emotion[i] in top_indices),
} for i in range(len(sample_indices))]
pd.DataFrame(sample_rows).to_csv(os.path.join(EXPORT_DIR, 'sample_points.csv'), index=False)
print('Saved: sample_points.csv')

# ---- 7. Per-emotion F1 for every compressed model -----------------------
per_emo_rows = [{
    'study': 'baseline', 'model': 'Baseline', 'rank': None, 'emotion': emo,
    'f1': float(baseline_per_emotion[i]),
} for i, emo in enumerate(emotion_names)]
for (label, rank), arr in comp_per_emotion.items():
    for i, emo in enumerate(emotion_names):
        per_emo_rows.append({'study': 'component', 'model': label, 'rank': rank,
                             'emotion': emo, 'f1': float(arr[i])})
for (label, rank), arr in depth_per_emotion.items():
    for i, emo in enumerate(emotion_names):
        per_emo_rows.append({'study': 'depth', 'model': label, 'rank': rank,
                             'emotion': emo, 'f1': float(arr[i])})
pd.DataFrame(per_emo_rows).to_csv(
    os.path.join(EXPORT_DIR, 'per_emotion_results.csv'), index=False,
)
print(f'Saved: per_emotion_results.csv  ({len(per_emo_rows)} rows)')

# ---- 8. t-SNE coordinates (component study) -----------------------------
tsne_comp_rows = []
for rank in RANKS:
    tsne_per_model, bounds = comp_tsne[rank]
    for name, coords in tsne_per_model.items():
        for i, (x, y) in enumerate(coords):
            tsne_comp_rows.append({
                'rank': rank, 'model': name, 'point_idx': i,
                'x': float(x), 'y': float(y),
                'dominant_emotion_idx': int(dominant_emotion[i]),
                'dominant_emotion': emotion_names[dominant_emotion[i]],
                'is_top_emotion': bool(dominant_emotion[i] in top_indices),
            })
pd.DataFrame(tsne_comp_rows).to_csv(
    os.path.join(EXPORT_DIR, 'tsne_component.csv'), index=False,
)
print(f'Saved: tsne_component.csv  ({len(tsne_comp_rows)} rows)')

# ---- 9. t-SNE coordinates (depth study) ---------------------------------
tsne_depth_rows = []
for rank in RANKS:
    tsne_per_model, bounds = depth_tsne[rank]
    for name, coords in tsne_per_model.items():
        for i, (x, y) in enumerate(coords):
            tsne_depth_rows.append({
                'rank': rank, 'model': name, 'point_idx': i,
                'x': float(x), 'y': float(y),
                'dominant_emotion_idx': int(dominant_emotion[i]),
                'dominant_emotion': emotion_names[dominant_emotion[i]],
                'is_top_emotion': bool(dominant_emotion[i] in top_indices),
            })
pd.DataFrame(tsne_depth_rows).to_csv(
    os.path.join(EXPORT_DIR, 'tsne_depth.csv'), index=False,
)
print(f'Saved: tsne_depth.csv  ({len(tsne_depth_rows)} rows)')

# ---- 10. t-SNE bounds (for regenerating plots) --------------------------
bounds_rows = []
for rank in RANKS:
    for study, tsne_dict in [('component', comp_tsne), ('depth', depth_tsne)]:
        _, b = tsne_dict[rank]
        bounds_rows.append({
            'study': study, 'rank': rank,
            'x_min': float(b[0]), 'x_max': float(b[1]),
            'y_min': float(b[2]), 'y_max': float(b[3]),
        })
pd.DataFrame(bounds_rows).to_csv(
    os.path.join(EXPORT_DIR, 'tsne_bounds.csv'), index=False,
)
print('Saved: tsne_bounds.csv')

# ---- 11. Master table (every evaluated model in one place) --------------
master_rows = [{
    'study': 'baseline', 'model': 'Baseline', 'rank': None, 'n_layers': 72,
    'f1_macro': baseline_f1, 'f1_retention': 1.0,
    'compression_ratio': 1.0, 'silhouette': None,
}]
for r in comp_results:
    master_rows.append({
        'study': 'component', 'model': r['component'], 'rank': r['rank'],
        'n_layers': r['n_layers'], 'f1_macro': r['f1_macro'],
        'f1_retention': r['f1_macro'] / baseline_f1,
        'compression_ratio': r['compression_ratio'],
        'silhouette': comp_silhouette[(r['component'], r['rank'])],
    })
for r in depth_results:
    master_rows.append({
        'study': 'depth', 'model': r['depth_group'], 'rank': r['rank'],
        'n_layers': r['n_layers'], 'f1_macro': r['f1_macro'],
        'f1_retention': r['f1_macro'] / baseline_f1,
        'compression_ratio': r['compression_ratio'],
        'silhouette': depth_silhouette[(r['depth_group'], r['rank'])],
    })
pd.DataFrame(master_rows).to_csv(
    os.path.join(EXPORT_DIR, 'sensitivity_master.csv'), index=False,
)
print(f'Saved: sensitivity_master.csv  ({len(master_rows)} rows)')

print(f'\nAll CSVs written to {EXPORT_DIR}')

## Summary

In [ ]:
figures = [
    'comp_scatter_rank256.png', 'comp_scatter_rank128.png', 'comp_scatter_rank64.png',
    'comp_kde_rank256.png',     'comp_kde_rank128.png',     'comp_kde_rank64.png',
    'depth_scatter_rank256.png','depth_scatter_rank128.png','depth_scatter_rank64.png',
    'depth_kde_rank256.png',    'depth_kde_rank128.png',    'depth_kde_rank64.png',
    'heatmap_f1_component.png', 'heatmap_silhouette_component.png',
    'heatmap_f1_depth.png',     'heatmap_silhouette_depth.png',
    'dashboard_compression_analysis.png', 'barchart_f1_retention.png',
]
csvs = [
    'component_sensitivity.csv', 'depth_sensitivity.csv',
    'component_f1_matrix.csv', 'component_silhouette_matrix.csv',
    'depth_f1_matrix.csv',     'depth_silhouette_matrix.csv',
    'baseline_info.csv',       'top_emotions.csv',
    'sample_points.csv',       'per_emotion_results.csv',
    'tsne_component.csv',      'tsne_depth.csv',
    'tsne_bounds.csv',         'sensitivity_master.csv',
]

print('=' * 65)
print('SUMMARY -- Phase 3: Component and depth sensitivity')
print('=' * 65)
print(f'Baseline F1 macro: {baseline_f1:.4f}')
print(f't-SNE subsample:   {N_SAMPLES}')
print(
    f'Models evaluated:  {1 + len(comp_results) + len(depth_results)} '
    f'(1 baseline + {len(comp_results)} component + {len(depth_results)} depth)'
)
print(f't-SNE runs:        {2 * len(RANKS)} ({len(RANKS)} component + {len(RANKS)} depth)')

print(f'\nFigures ({len(figures)}):')
for fig_name in figures:
    status = 'ok' if os.path.exists(os.path.join(FIGURES_DIR, fig_name)) else 'MISSING'
    print(f'  [{status}] {fig_name}')

print(f'\nCSVs in {EXPORT_DIR} ({len(csvs)}):')
for csv_name in csvs:
    status = 'ok' if os.path.exists(os.path.join(EXPORT_DIR, csv_name)) else 'MISSING'
    print(f'  [{status}] {csv_name}')